In [ ]:
import sys
sys.path.append('..')

In [ ]:
from core.graph import KnowledgeGarden
from retrieval.traversal import *
from datetime import datetime

In [ ]:
kg = KnowledgeGarden()

In [ ]:
kg.add_node("alice", "person", "Alice")
kg.add_node("bob", "person", "Bob")
kg.add_node("infra_team", "team", "Infrastructure Team")
kg.add_node("platform_team", "team", "Platform Team")
kg.add_node("auth_service", "service", "Authentication Service")
kg.add_node("payment_service", "service", "Payment Service")
kg.add_node("api_gateway", "service", "API Gateway")
kg.add_node("postgres", "tool", "Postgres Database")
kg.add_node("incident_001", "concept", "Incident 001")

In [ ]:
kg.add_edge("alice", "infra_team", "MEMBER_OF", "Alice is a member of the infrastructure team", datetime(2025, 1, 10))
kg.add_edge("bob", "platform_team", "MEMBER_OF", "Bob is a member of the platform team", datetime(2025, 1, 10))
kg.add_edge("alice", "platform_team", "MEMBER_OF", "Alice is a member of the platform team", datetime(2025, 3, 1))
kg.add_edge("infra_team", "auth_service", "OWNS", "Infrastructure team owns the auth service", datetime(2025, 1, 10))
kg.add_edge("platform_team", "payment_service", "OWNS", "Platform team owns the payment service", datetime(2025, 1, 10))
kg.add_edge("platform_team", "api_gateway", "OWNS", "Platform team owns the API gateway", datetime(2025, 1, 10))
kg.add_edge("auth_service", "postgres", "DEPENDS_ON", "Auth service depends on Postgres", datetime(2025, 1, 10))
kg.add_edge("payment_service", "postgres", "DEPENDS_ON", "Payment service depends on Postgres", datetime(2025, 1, 10))
kg.add_edge("api_gateway", "auth_service", "DEPENDS_ON", "API gateway depends on auth service", datetime(2025, 1, 10))
kg.add_edge("alice", "incident_001", "RESPONDED_TO", "Alice responded to incident 001", datetime(2025, 2, 20))
kg.add_edge("auth_service", "incident_001", "INVOLVED_IN", "Auth service was involved in incident 001", datetime(2025, 2, 20))

In [ ]:
# direct look up
auth_service = kg.nodes["auth_service"]
result = direct_lookup(kg, auth_service, "OWNS", direction = 'in')
for edge in result:
    print(edge.source, edge.relation, edge.target, edge.fact)

In [ ]:
# neighbor expansion
infra_team = kg.nodes["infra_team"]
result = neighbor_expansion(kg, infra_team, depth = 2)
for edge in result:
    print(edge.source, edge.relation, edge.target)

In [ ]:
# path finding
alice = kg.nodes["alice"]
postgres = kg.nodes["postgres"]
result = path_finding(kg, alice, postgres)
for edge in result:
    print(edge.source, edge.relation, edge.target)

In [ ]:
#impact traversal
postgres = kg.nodes["postgres"]
result = impact_traversal(kg, postgres, "DEPENDS_ON", direction='in')
for edge in result:
    print(edge.source, edge.relation, edge.target)

In [ ]:
# history traversal
alice = kg.nodes["alice"]
result = history_traversal(kg, alice, "MEMBER_OF", direction='out')
for edge in result:
    print(edge.source, edge.relation, edge.target, edge.valid_from, edge.valid_until)

In [ ]:
from retrieval.query import query
from llm_clients import LLMClient

In [ ]:
client = LLMClient()

In [ ]:
result = query(kg, "Who owns the auth service?", client)
print(result)

In [ ]:
from prompts import *
from retrieval.query_schema import *
from enrichment.resolver import *

In [ ]:
prompt = QUERY_INTENT_PROMPT.replace("{question}", "Who owns the auth service?")
response = client.generate_gemini(prompt, schema_type=QueryIntent)
print(response)

In [ ]:
query_intent = QueryIntent.model_validate_json(response)
print(query_intent)

In [ ]:
normalized_name = normalize(query_intent.anchor_entity)
print(normalized_name)

In [ ]:
for existing_node in kg.nodes.values():
    print(existing_node.name, normalize(existing_node.name))